# FH-01 · Environment, baseline and actionability (Phases 0, 1, 3, 4)
**Runtime → Change runtime type → T4 GPU.** Run cells top to bottom.

What this notebook does
1. Phase 0 - smoke test (load, determinism, VRAM, versions recorded)
2. Dataset checks, split by scenario group, **locked test hash**
3. Blind-review agreement (you vs. the drafted labels) and adjudication
4. Phase 1 - baseline conditions A0 (original prompt), A1 (improved prompt), B (few-shot), C (thinking, optional)
5. Phase 3 - actionability: confusion matrix, **false-action rate**, slices, minimal pairs, calibration
6. Phase 4 - structured output validity, entities, relative-time resolution

**Rule:** tune prompts on **dev** only. The **test** split is run once, at the end of tuning.
Colab is cloud - only synthetic/public data goes in here.

In [ ]:
# ---- Colab setup: upload futurehuman_testing_suite.zip when asked (once per session) ----
import os, sys, subprocess
SUITE = "/content/futurehuman_testing_suite"
if not os.path.isdir(SUITE):
    from google.colab import files
    up = files.upload()
    subprocess.run(["unzip", "-o", "-q", list(up)[0], "-d", "/content"], check=True)
os.chdir(SUITE); sys.path.insert(0, SUITE)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "accelerate", "bitsandbytes", "tabulate"])
print("ready in", os.getcwd())

In [ ]:
TAG = "base"          # change to "stage1" / "stage2" when you test a trained adapter (same notebook, same tests)
ADAPTER = None        # e.g. "/content/drive/MyDrive/adapters/stage1"  (None = untouched base model)
QUANT = "4bit"        # 4bit | 8bit | fp16
from fh import env
from fh.model import load_qwen, make_gen
env.set_seed(0)
model, tok = load_qwen("Qwen/Qwen3-4B", quant=QUANT, adapter=ADAPTER)
gen = make_gen(model, tok)
info = env.save_snapshot(f"results/env_{TAG}.json", model=model, model_id="Qwen/Qwen3-4B", quant=QUANT, seed=0,
                         extra=dict(adapter=ADAPTER))
info

## Phase 0 - smoke test

In [ ]:
import torch
probes = [[{"role": "user", "content": p}] for p in ["Say hi in one word.", "What is 2+2?", "Name a colour.", "Capital of India?", "Say OK."]]
a = [gen(m, max_new_tokens=20)["text"] for m in probes]
b = [gen(m, max_new_tokens=20)["text"] for m in probes]
print("deterministic (greedy, run twice):", a == b)
r = gen(probes[0], max_new_tokens=50); print({k: r[k] for k in ("latency_s", "n_in", "n_out")})
print("peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))

## Dataset: integrity, splits, locked test hash

In [ ]:
import os, json
from fh.dataset import *
DATA = "data/actionability_v1_adjudicated.jsonl" if os.path.exists("data/actionability_v1_adjudicated.jsonl") else "data/actionability_v1.jsonl"
cases = load_actionability(DATA)
print("using", DATA)
print(json.dumps(validate_dataset(cases), indent=1, ensure_ascii=False))
dev = [c for c in cases if c["split"] == "dev"]; test = [c for c in cases if c["split"] == "test"]
print(len(dev), "dev /", len(test), "test")
h = freeze_hash(cases); open(f"results/TEST_HASH_{TAG}.txt", "w").write(h); print("TEST HASH", h)

## Blind review → agreement (do this BEFORE the locked test run)
1. Open `data/review_sheet.csv` in Excel/Sheets. Read `GUIDELINE.md`. Fill **your_label** for every row **without looking at any labels**.
2. Upload the filled file below. You get agreement + Cohen's kappa between you and the drafted labels, and a list of disagreements.
3. For each disagreement decide the final label (the guideline may need fixing too) and put it in `FINAL`.
Skip this cell block only if you have already adjudicated (the adjudicated file is then picked up automatically).

In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()
r = compare_review(cases, list(up)[0])
print(r["n_labeled"], "of", r["n_total"], "labelled | agreement", round(r["agreement"], 3), "| kappa", round(r["kappa"], 3), "(rule of thumb: >= 0.7)")
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(r["disagreements"])[["id", "gold", "human", "conversation", "note", "comment"]]

In [ ]:
FINAL = {}   # e.g. {"A049": "ambiguous", "A055": "non_actionable"}  - your decision for each disagreement
cases = apply_adjudication(cases, FINAL)
with open("data/actionability_v1_adjudicated.jsonl", "w", encoding="utf-8") as f:
    for c in cases: f.write(json.dumps(c, ensure_ascii=False) + "\n")
cases = load_actionability("data/actionability_v1_adjudicated.jsonl")
dev = [c for c in cases if c["split"] == "dev"]; test = [c for c in cases if c["split"] == "test"]
h = freeze_hash(cases); open(f"results/TEST_HASH_{TAG}.txt", "w").write(h); print("LOCKED TEST HASH", h, "| changed labels:", len(FINAL))

## Phase 1 - baseline on DEV (iterate here)

In [ ]:
from fh.evaluate import run_cases, save_run
from fh.report import print_summary
from fh.prompts import select_fewshot
dev_A0 = run_cases(dev, gen, "A0_original"); print_summary(dev_A0, "DEV  A0 original prompt")

In [ ]:
dev_A1 = run_cases(dev, gen, "A1_improved"); print_summary(dev_A1, "DEV  A1 improved prompt")

In [ ]:
fs = select_fewshot(dev, n=(3, 3, 2), seed=0)
print("few-shot ids (dev only, excluded from scoring):", [c["id"] for c in fs])
dev_B = run_cases(dev, gen, "A1_improved", fewshot=fs); print_summary(dev_B, "DEV  B few-shot")
for k, d in {"A0": dev_A0, "A1": dev_A1, "B": dev_B}.items():
    save_run(d, f"{TAG}_{k}_dev", dict(prompt=k, split="dev"))

### Look at the errors before touching the prompt
The auto-tags below map to the error taxonomy in the plan. If you edit `PROMPT_V1` in `fh/prompts.py`, **rename it** (e.g. `A2_...`) and note what changed - never overwrite silently.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 160)
def auto_tag(r):
    if r.pred_label == "invalid": return "invalid_json_or_schema"
    if r.gold_label != "actionable" and r.pred_label == "actionable": return "OVER_ACTION (false action)"
    if r.gold_label == "actionable" and r.pred_label == "non_actionable": return "under_action (missed)"
    if r.gold_label == "actionable" and r.pred_label == "ambiguous": return "unnecessary_ask"
    if r.gold_label == "ambiguous" and r.pred_label == "non_actionable": return "ambiguity_ignored"
    if r.gold_label == "non_actionable" and r.pred_label == "ambiguous": return "over_ask"
    if r.gold_label == "actionable" and not r.intent_ok: return "wrong_intent"
    if r.gold_label == "actionable" and (r.entity_recall or 0) < 0.5: return "wrong_entity"
    return "ok"
dev_B["tag"] = dev_B.apply(auto_tag, axis=1)
print(dev_B["tag"].value_counts())
dev_B[dev_B.tag != "ok"][["id", "lang", "gold_label", "pred_label", "pred_intent", "tag", "raw"]].sort_values("tag")

## Phase 1/3 - locked TEST run (after dev tuning is finished)
Runs A0, A1 and B on the test split and saves raw predictions. **Do not change prompts after looking at these numbers.**

In [ ]:
RUNS = {}
conds = {"A0_original": dict(prompt_id="A0_original"), "A1_improved": dict(prompt_id="A1_improved"),
         "B_fewshot": dict(prompt_id="A1_improved", fewshot=fs)}
for rid, kw in conds.items():
    df = run_cases(test, gen, **kw); RUNS[rid] = df
    save_run(df, f"{TAG}_{rid}_test", dict(kw, fewshot=[c["id"] for c in kw.get("fewshot", [])], model="Qwen3-4B", quant=QUANT, adapter=ADAPTER, env=info))
    print_summary(df, f"TEST {rid}")

In [ ]:
# Condition C (optional): thinking mode. Qwen recommends SAMPLING here (greedy can loop) so results are noisier and ~10x slower.
RUN_C = False
if RUN_C:
    sub = test[:40]
    dfC = run_cases(sub, gen, "A1_improved", fewshot=fs, thinking=True, max_new_tokens=1024)
    RUNS["C_thinking_40"] = dfC; save_run(dfC, f"{TAG}_C_thinking_40_test", dict(thinking=True, n=40)); print_summary(dfC, "TEST C thinking (first 40)")

## Phase 3 - analysis of the best condition

In [ ]:
BEST = "B_fewshot"          # choose from RUNS (decide by DEV results, not test)
df = RUNS[BEST]
from fh.metrics import risk_coverage, confusion
import matplotlib.pyplot as plt
print(df.assign(tag=df.apply(auto_tag, axis=1)).tag.value_counts())
print("\nMinimal pairs - groups with ANY wrong variant:")
bp = df[df.pair.notna()].groupby("group").filter(lambda x: (x.gold_label != x.pred_label).any())
bp[["pair", "id", "gold_label", "pred_label", "raw"]].sort_values(["pair", "id"])

In [ ]:
rc = risk_coverage(df); print(rc.to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(rc.coverage, rc.error_rate, "o-"); ax[0].set_xlabel("coverage (share of actionable cases executed)"); ax[0].set_ylabel("error rate of executed"); ax[0].set_title("Risk-coverage")
import numpy as np
ok = (df.gold_label == df.pred_label).astype(float); c = df.pred_conf.fillna(.5)
bins = np.linspace(0, 1, 11); idx = np.digitize(c, bins) - 1
xs = [c[idx == i].mean() for i in range(10) if (idx == i).any()]; ys = [ok[idx == i].mean() for i in range(10) if (idx == i).any()]
ax[1].plot([0, 1], [0, 1], "--", c="gray"); ax[1].plot(xs, ys, "o-"); ax[1].set_xlabel("confidence"); ax[1].set_ylabel("accuracy"); ax[1].set_title("Reliability (calibration)")
plt.tight_layout(); plt.savefig(f"results/{TAG}_risk_calibration.png", dpi=150); plt.show()
print("distinct confidence values:", df.pred_conf.nunique(), "-> if this is ~1-2 the confidence field cannot drive the gate")

## Phase 4 - structured output, entities, time resolution

In [ ]:
errs = df.schema_errors.fillna("").replace("", pd.NA).dropna().str.split(";").explode().value_counts()
print("schema error types:\n", errs if len(errs) else "none")
print("\nintent accuracy on gold-actionable:", df[df.gold_label == "actionable"].intent_ok.mean().round(3))
print("entity recall / precision (actionable):", df.entity_recall.mean().round(3), df.entity_precision.mean().round(3))
print("\nintents the model produced for gold-actionable, wrong ones:")
df[(df.gold_label == "actionable") & (~df.intent_ok)][["id", "gold_intents", "pred_intent", "raw"]]

In [ ]:
from fh import timeres
tr = pd.DataFrame(timeres.run(gen)); tr.to_csv(f"results/timeres_{TAG}.csv", index=False)
print("exact-match", tr.ok.mean().round(3), "| 'kal/parso/next Monday' must be AMBIGUOUS:")
tr[["id", "text", "gold", "pred", "ok"]]

In [ ]:
from fh.report import update_capabilities, caps_from_df
caps = update_capabilities(TAG, **caps_from_df(df), timeres_exact=dict(value=float(tr.ok.mean())))
print("saved results/capabilities_%s.json" % TAG); caps

**Next:** FH-02 (MultiWOZ + MASSIVE), FH-03 (gate, simulated tools, memory, privacy). They read the predictions saved in `results/`.